# Identity Gate

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2024 Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
import torch
from IPython.display import Markdown, SVG, display
from sympy import Symbol

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import identity_cost_key
from flagquantum.compiler.registers.primitives.identity import register_identity_ir
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg

## `Identity`
The identity gate on `n` qubits.

#### Parameters
 - `bitsize`: number of qubits `n`, defaults to 1. 

#### Registers
 - `q`: register of `n` qubits


In [ ]:
def identity_example(name, width):
    return name, Signature.build(q=width), identity_cost_key(width)

### Example Instances

In [ ]:
n = Symbol("n")
identity_symb = identity_example("Identity(n)", n)

In [ ]:
identity = identity_example("Identity", 1)

In [ ]:
identity_n = identity_example("Identity(4)", 4)

#### Graphical Signature

In [ ]:
examples = (identity_symb, identity, identity_n)
display(SVG(field_signature_svg([(name, signature, {}) for name, signature, _ in examples])))

### Call Graph

In [ ]:
name, _, key = identity_symb
graph = symbolic_call_graph(key, {}, max_depth=1)
assert graph.direct_calls == {} and graph.leaf_counts == {key: 1}
display(SVG(render_call_graph_svg(export_call_graph_dot(key, graph.direct_calls, title=lambda _: name))))
display(Markdown(format_call_graph_markdown(key, graph.direct_calls, graph.leaf_counts, title=lambda _: name)))

### Bind and append

The concrete register is through-going. The four-bit example appends one identity instruction per bound wire in register order. The symbolic example stays in the signature and resource graph until concrete wires are bound. The source models the whole register as one atomic leaf with zero QEC gates.

In [ ]:
circuit = Circuit(4, dtype=torch.complex128).h(0).cx(0, 3).s(3)
initial = circuit.state().clone()
op = FieldOperation(4, register_identity_ir((3, 1, 2, 0)))
circuit.append(op)
torch.testing.assert_close(circuit.state(), initial, rtol=0, atol=1e-12)
print("Emitted", [(gate.name, gate.wires) for gate in op.instructions])
print("Source QEC model", query_qec_gate_cost(identity_n[2]))